<a href="https://colab.research.google.com/github/Manoj-Kumar-Ejjada-AI/models/blob/manoj_models/GenAI/Masked_Self_Attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [132]:
class MaskedSelfAttention(nn.Module):

  def __init__(self,
               model_dim =2,
               row_dim = 0,
               col_dim = 1):

    super().__init__()

    self.row_dim = row_dim
    self.col_dim = col_dim

    self.w_q = nn.Linear(in_features=model_dim, out_features=model_dim, bias=False)
    self.w_k = nn.Linear(in_features=model_dim, out_features=model_dim, bias=False)
    self.w_v = nn.Linear(in_features=model_dim, out_features=model_dim, bias=False)

  def Forward(self, PE_Embeddings, mask = None):

    q = self.w_q(PE_Embeddings)
    k = self.w_k(PE_Embeddings)
    v = self.w_v(PE_Embeddings)


    similarities = torch.matmul(q, k.transpose(dim0 = self.row_dim, dim1 = self.col_dim))
    scaled_similarities = similarities/(torch.tensor(k.size(self.col_dim)**0.5))
    if mask is not None:
      scaled_similarities = scaled_similarities.masked_fill(mask = mask, value = float('-inf'))
    attention_percents = F.softmax(scaled_similarities,dim=self.col_dim)
    attention_scores = torch.matmul(attention_percents,v)

    return attention_scores



In [133]:
torch.manual_seed(42)

embed_inputs = torch.tensor([
    [1.16,0.23],
    [0.57,1.36],
    [4.41,-2.16]
])
masked_at = MaskedSelfAttention()
print("Self Attention: ", masked_at.Forward(embed_inputs))
mask = torch.ones(3,3)  # As input has three batches
mask = torch.tril(mask)
mask = mask == 0
# print(mask)
print("Masked Self Attenction", masked_at.Forward(embed_inputs,mask))

Self Attention:  tensor([[1.0100, 1.0641],
        [0.2040, 0.7057],
        [3.4989, 2.2427]], grad_fn=<MmBackward0>)
Masked Self Attenction tensor([[ 0.6038,  0.7434],
        [-0.0062,  0.6072],
        [ 3.4989,  2.2427]], grad_fn=<MmBackward0>)
